# Bibliotecas

In [1]:
# Configurações gerais
import warnings

warnings.filterwarnings("ignore")

# Manipulação de dados
import numpy as np
import pandas as pd
import joblib

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)

# Visualização
import matplotlib.pyplot as plt
import seaborn as sns

# Modelo
from xgboost import XGBClassifier

# Validação e seleção de modelos
from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate
)

# Métricas de avaliação
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score
)

# Funções

In [ ]:
## Criando Features
def create_features(df):
    df = df.copy()

    cols_to_round = ["FCVC"]

    for col in cols_to_round:
        if col in df.columns:
            df[col] = df[col].round().astype(int)

    df["flag_female"] = df["Gender"].map({"Female": 1, "Male": 0})
    df = df.drop(columns=["Gender"])

    df["flag_family_history"] = df["family_history"].map({"yes": 1, "no": 0})
    df = df.drop(columns=["family_history"])

    mapa_frequencia = {
        "no": 0,
        "Sometimes": 1,
        "Frequently": 2,
        "Always": 3
    }

    df["CAEC_freq"] = df["CAEC"].map(mapa_frequencia)
    df["CALC_freq"] = df["CALC"].map(mapa_frequencia)

    df = df.drop(columns=["CAEC", "CALC"])

    df["MTRANS_Public_Transportation"] = (
        df["MTRANS"] == "Public_Transportation"
    ).astype(int)

    # IMC
    df["BMI"] = df["Weight"] / (df["Height"] ** 2)

    return df

# Exemplo: df_fe = create_features(df)

In [27]:
def calcular_metricas(nome, y_true, y_pred, y_proba):
    y_true = y_true.squeeze()

    return {
        "Conjunto": nome,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision macro": precision_score(
            y_true, y_pred,
            average="macro",
            zero_division=0
        ),
        "F1 macro": f1_score(
            y_true, y_pred,
            average="macro",
            zero_division=0
        ),
        "Sensibilidade macro": recall_score(
            y_true, y_pred,
            average="macro",
            zero_division=0
        ),
        "ROC AUC OVR macro": roc_auc_score(
            y_true,
            y_proba,
            multi_class="ovr",
            average="macro"
        )
    }

# Importando o modelo

In [ ]:
artefato_carregado = joblib.load(
    "modelo_xgboost_obesidade.joblib"
)

modelo_carregado = artefato_carregado["modelo"]
target_map = artefato_carregado["target_map"]
colunas_input = artefato_carregado["colunas_input"]
valores_fillna = artefato_carregado["valores_fillna"]
metricas_treinamento = artefato_carregado["metricas_treinamento"]

# Importando base

In [38]:
CSV_PATH = "Obesity.csv"

df = pd.read_csv(CSV_PATH)

target = "Obesity"

# Verificador
assert not df.empty, "Erro: o DataFrame está vazio."

# Preparação

In [39]:
# Padronizar nomes das colunas
df.columns = df.columns.str.strip()

# Padronizar valores textuais sem transformar NA em "nan"
for col in df.select_dtypes(include=["object", "string"]).columns:
    df[col] = df[col].astype("string").str.strip()
    df[col] = df[col].replace("", pd.NA)

# Preencher valores ausentes
df.fillna(value=valores_fillna, inplace=True)

# Remover duplicidades
df.drop_duplicates(inplace=True)

# Verificar se existem valores nulos
colunas_com_nulos = df.columns[df.isna().any()].tolist()
assert not colunas_com_nulos, (
    f"Erro: existem valores nulos nas colunas: {colunas_com_nulos}"
)

In [40]:
# Aplicar o mapeamento da variavel target
df["target_numerico"] = df[target].map(target_map)

# Checar se alguma classe ficou sem mapeamento
if df["target_numerico"].isna().any():
    print("Atenção: existem classes da target que não foram mapeadas.")
    print(df.loc[df["target_numerico"].isnull(), "target_original"].unique())
else:
    # print("Mapeamento da target realizado com sucesso.")
    pass

# Garantir tipo inteiro
df["target_numerico"] = df["target_numerico"].astype(int)

## Feature Eng

In [ ]:
df = create_features(df)

## Separando Variaveis de Input e Target

In [29]:
X = df.drop(columns=[target,"target_numerico"])
X = X[colunas_input].copy()

assert X.columns.tolist() == colunas_input, (
    "Erro: as colunas ou a ordem do DataFrame são diferentes "
    "das utilizadas no treinamento."
)

y_real = df["target_numerico"]

# Predição

In [30]:
y_predicao = modelo_carregado.predict(X)
y_probabilidades = modelo_carregado.predict_proba(X)

In [31]:
resultados = [
    calcular_metricas(
        "Novos dados",
        y_real,
        y_predicao,
        y_probabilidades
    )
]

tabela_metricas = (
    pd.DataFrame(resultados)
    .set_index("Conjunto")
    .round(3)
)

display(tabela_metricas)

,Accuracy,Precision macro,F1 macro,Sensibilidade macro,ROC AUC OVR macro
Conjunto,,,,,
Novos dados,0.989,0.989,0.989,0.989,0.999


In [37]:
target_map_invertido = {
    codigo: nome
    for nome, codigo in target_map.items()
}

classes = modelo_carregado.classes_

relatorio_dict = classification_report(
    y_real,
    y_predicao,
    labels=classes,
    target_names=[target_map_invertido[classe] for classe in classes],
    output_dict=True,
    zero_division=0
)

df_classification_report = (
    pd.DataFrame(relatorio_dict)
    .T
    .round(4)
)

resultado_corte = (
    df_classification_report
    .loc[target_map_invertido.values(), ["precision", "recall"]]
    .assign(atingiu_corte_recall=lambda x: x["recall"] >= 0.75)
)

display(resultado_corte)

,precision,recall,atingiu_corte_recall
Insufficient_Weight,1.0000,0.9888,True
Normal_Weight,0.9860,1.0000,True
Overweight_Level_I,0.9854,0.9783,True
Overweight_Level_II,0.9827,0.9793,True
Obesity_Type_I,0.9887,0.9972,True
Obesity_Type_II,0.9866,0.9899,True
Obesity_Type_III,0.9969,0.9907,True
